In [2]:
!pip install -q SpeechRecognition deep-translator gTTS
!apt-get install -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.8 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [5]:
import ipywidgets as widgets
from IPython.display import display, Javascript, Audio
from google.colab.output import eval_js
from base64 import b64decode
import speech_recognition as sr
from deep_translator import GoogleTranslator
from gtts import gTTS
import os

# --- JavaScript for Unlimited Recording with Advanced Noise Cancellation ---
RECORD_JS = """
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = () => new Promise(async resolve => {
  // Added Noise Suppression, Echo Cancellation & Auto Gain Control
  const audioConstraints = {
      echoCancellation: true,
      noiseSuppression: true,
      autoGainControl: true
  };
  stream = await navigator.mediaDevices.getUserMedia({ audio: audioConstraints })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()

  // Stop Button UI
  let btn = document.createElement('button');
  btn.innerHTML = '🛑 Click Here to Stop Recording';
  btn.style.padding = '10px 20px';
  btn.style.color = 'white';
  btn.style.backgroundColor = '#dc3545';
  btn.style.border = 'none';
  btn.style.borderRadius = '5px';
  btn.style.fontSize = '16px';
  btn.style.cursor = 'pointer';
  btn.style.marginTop = '10px';
  btn.style.fontWeight = 'bold';
  btn.style.boxShadow = '0px 4px 6px rgba(0,0,0,0.2)';
  document.body.appendChild(btn);

  btn.onclick = async () => {
    recorder.stop()
    btn.remove()
  }

  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
})
"""

def record_audio():
    display(Javascript(RECORD_JS))
    print("🎤 Recording started! Scroll down slightly to find the red 🛑 Stop button.")
    s = eval_js('record()')
    b = b64decode(s.split(',')[1])
    with open('audio.webm','wb') as f:
        f.write(b)
    os.system("ffmpeg -y -i audio.webm -ac 1 -ar 16000 audio.wav -loglevel quiet")
    return 'audio.wav'

# --- UI Setup ---
out_stt = widgets.Output()
out_tts = widgets.Output()

# HTML Template for beautiful output box
HTML_TEMPLATE = """
<div style="background-color: #1e1e1e; padding: 15px; border-radius: 8px; border: 1px solid #444; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin-top: 10px; box-shadow: inset 0px 0px 10px rgba(0,0,0,0.5);">
    <div style="margin-bottom: 10px;">
        <span style="color: #FF9800; font-size: 12px; font-weight: bold; text-transform: uppercase;">🎙️ Original Voice Text:</span><br>
        <span style="color: #E0E0E0; font-size: 16px;">{original}</span>
    </div>
    <hr style="border-top: 1px dashed #555; margin: 10px 0;">
    <div>
        <span style="color: #03A9F4; font-size: 12px; font-weight: bold; text-transform: uppercase;">🌐 Translated Text:</span><br>
        <span style="color: #4CAF50; font-size: 16px; font-weight: bold;">{translated}</span>
    </div>
</div>
"""

# --- Feature 1: Speech to Text & Translation ---
stt_title = widgets.HTML("<h3>1. Speech to Text & Translation</h3>")
stt_record_btn = widgets.Button(description="Start Recording", button_style='danger', icon='microphone')
stt_lang_dropdown = widgets.Dropdown(options=[('Bangla to English', 'en'), ('English to Bangla', 'bn')], description='Translate to:')

# Using HTML widget instead of Textarea for better styling
stt_html_out = widgets.HTML(value='<div style="color: #888; font-style: italic;">Result will appear here...</div>')

def on_record_click(b):
    with out_stt:
        out_stt.clear_output()
        stt_html_out.value = '<div style="color: #FFA500; font-weight: bold;">⏳ Recording in progress... Waiting for you to click stop.</div>'
        filename = record_audio()

        r = sr.Recognizer()
        # Adjusting for ambient noise locally as well
        with sr.AudioFile(filename) as source:
            r.adjust_for_ambient_noise(source, duration=0.5)
            audio_data = r.record(source)
            try:
                input_lang = 'bn-BD' if stt_lang_dropdown.value == 'en' else 'en-US'
                stt_html_out.value = '<div style="color: #00BFFF; font-weight: bold;">🔄 Processing and Recognizing speech...</div>'

                text = r.recognize_google(audio_data, language=input_lang)

                stt_html_out.value = '<div style="color: #00BFFF; font-weight: bold;">🌐 Translating...</div>'
                translated_text = GoogleTranslator(source='auto', target=stt_lang_dropdown.value).translate(text)

                # Applying the custom UI template
                stt_html_out.value = HTML_TEMPLATE.format(original=text, translated=translated_text)
                print("✅ Processing Complete!")

            except sr.UnknownValueError:
                stt_html_out.value = '<div style="color: #F44336; font-weight: bold;">⚠️ Sorry, could not clearly understand the audio. Please speak a bit louder or closer.</div>'
            except Exception as e:
                stt_html_out.value = f'<div style="color: #F44336;">Error: {e}</div>'

stt_record_btn.on_click(on_record_click)

# --- Feature 2: Text to Speech (Accent Based) ---
tts_title = widgets.HTML("<hr><h3>2. Text to Speech</h3>")
tts_input = widgets.Text(description='Text:', placeholder='Type text here...', layout=widgets.Layout(width='70%'))
tts_lang_dropdown = widgets.Dropdown(options=[('English Accent', 'en'), ('Bangla Accent', 'bn')], description='Text Accent:')
tts_btn = widgets.Button(description="Generate & Play", button_style='success', icon='play')

def on_tts_click(b):
    with out_tts:
        out_tts.clear_output()
        if not tts_input.value:
            print("⚠️ Please enter some text first!")
            return

        print(f"🔊 Generating Audio with {tts_lang_dropdown.label}...")
        tts = gTTS(text=tts_input.value, lang=tts_lang_dropdown.value)
        tts.save('output.mp3')

        display(Audio('output.mp3', autoplay=True))
        print("✅ Audio ready!")

tts_btn.on_click(on_tts_click)

# --- Display Interface ---
display(stt_title, widgets.HBox([stt_record_btn, stt_lang_dropdown]), out_stt, stt_html_out)
display(tts_title, widgets.HBox([tts_input, tts_lang_dropdown, tts_btn]), out_tts)

HTML(value='<h3>1. Speech to Text & Translation</h3>')

Output()

HTML(value='<div style="color: #888; font-style: italic;">Result will appear here...</div>')

HTML(value='<hr><h3>2. Text to Speech</h3>')

Output()